In [12]:
!pip install -q transformers torch langdetect


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\furyd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face API Token: ")
hf_token = os.environ["HF_TOKEN"]

In [ ]:
# Importing the necessary libraries
import re
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langdetect import detect
from huggingface_hub import login()
login(token=hf_token)

In [15]:
# Initialize Translation class
class NotebookTranslator:
    def __init__(self):
        self.model_name = "google/gemma-2-2b-it"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )

    def detect_language(self, text):
        try:
            if not text or not any(char.isalpha() for char in text):
                return "en"
            return detect(text)
        except Exception:
            return "en" 

    def translate(self, text, source='auto', target='en'):
        src_lang_code = self.LANG_MAP.get(source, "eng_Latn") 
        tgt_lang_code = self.LANG_MAP.get(target, "eng_Latn") 
        self.tokenizer.src_lang = src_lang_code
        inputs = self.tokenizer(text, return_tensors="pt")
        tgt_lang_id = self.tokenizer.convert_tokens_to_ids(tgt_lang_code)
        
        translated_tokens = self.model.generate(
            **inputs, forced_bos_token_id=tgt_lang_id, max_length=250
        )
        return self.tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

# Instantiate engines
translator = NotebookTranslator()
generator = pipeline("text-generation", 
    model="google/gemma-2-2b-it",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto")

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
401 Client Error. (Request ID: Root=1-6a11cec0-5ae19c645acab5bc559dc3bf;06f86470-bd14-4624-9c5a-665fcbfde918)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
def extract_questions(text):
    return re.findall(r'\d+\..*?\?', text) or [text]

def generate_ai_response(prompt, max_new_tokens=350):
    messages = [{"role": "user", "content": prompt}]
    
    # Generate the text completion
    res = generator(messages, max_new_tokens=max_new_tokens, temperature=0.5, do_sample=True)
    
    # Pull out just the newly generated response text content from the output pipeline
    return res[0]['generated_text'][-1]['content']

In [ ]:
print("\n" + "="*40)
enquiry_text = input("Enter your enquiry: ").strip()  
# Added target language preference input
target_pref = input("Enter desired target language code (e.g., 'en', 'fr', 'es', 'de'): ").strip().lower()
print("="*40)

In [ ]:
# DETECT SOURCE LANGUAGE 
detect_lang = translator.detect_language(enquiry_text)
translated_enq = translator.translate(enquiry_text, source=detect_lang, target=target_pref)

# Print results
print(f"Original Enquiry: {enquiry_text}")
print(f"Detected Language: {detect_lang.upper()}")
print(f"Translated Response Target ({target_pref.upper()}): {translated_enq}")

Original Enquiry: Hi, I'm Daniel! I was wondering what the capital of Angola was?
Detected Language: EN
Translated Response Target (DE): Ich frage mich, was die Hauptstadt von Angola ist.


In [ ]:
# QUESTION SEARCH
def contains_question(text):
    return '?' in text.strip()

print(f"Contains Question? {contains_question(enquiry_text)}")

Contains Question? True


In [ ]:
# Generate and display responses

# Enquiry in English
enquiry_en = translator.translate(enquiry_text, source=detect_lang, target='en')

# Response in English
ai_response = generate_ai_response(enquiry_en, max_new_tokens=150)
print(f"AI Response (Internal English): {ai_response}\n")

# Re-translates back to user's preferred language
response_transl = translator.translate(ai_response, source='en', target=target_pref)
print(f"AI Chatbot Response ({target_pref.upper()}): {response_transl}")

AI Response (Internal English): I was wondering what the capital of Angola was?    Editions.  

AI Chatbot Response (DE): Ich fragte mich, was die Hauptstadt von Angola war.
